# Section A

#### Project Architecture, Data Loading & Relational Merging

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

### Q no.1 Set up your project folder structure with the following directories: data/, notebooks/, app/, assets/. Write a Python script that loads all 9 CSV files into a dictionary of DataFrames using a loop and prints each DataFrame's name, shape, and memory usage in MB using .memory_usage(deep=True).sum().

In [ ]:
def read_multiple_csv(folder_path, file_pattern = "*.csv"):
    
    file_paths = glob.glob(os.path.join(folder_path, file_pattern))

    if not file_paths:
        raise FileNotFoundError(f"No files found in {folder_path} matching {file_pattern}")

    dataframes = {}
    for file in file_paths:
        try:
            df = pd.read_csv(file)
            filename = os.path.basename(file)
            dataframes[filename] = df
        except Exception as e:
            print(f"Error reading {file}: {e}")

    return dataframes

In [ ]:
folder = "../data"
dfs = read_multiple_csv(folder)


In [ ]:
for df_name in dfs.keys():
    print(f"Dataframe name = {df_name}")
    print(f"Shape = {dfs[df_name].shape}")
    print(f"Used memory = {(dfs[df_name].memory_usage(deep=True).sum()/1024**2).round(2)} MB \n")

### Q no.2 Perform a multi-table merge to build a single master DataFrame called df_master by joining: orders → order_items → products → customers → sellers → payments → reviews. Use appropriate join types (left/inner) for each step. State the final shape and list any columns that appear more than once after merging (suffixed columns).

In [ ]:
order = dfs['olist_orders_dataset.csv']
order_item = dfs['olist_order_items_dataset.csv']
products = dfs['olist_products_dataset.csv']
customers = dfs['olist_customers_dataset.csv']
sellers = dfs['olist_sellers_dataset.csv']
payments = dfs['olist_order_payments_dataset.csv']
review = dfs['olist_order_reviews_dataset.csv']


In [ ]:
df_master = order.merge(order_item, on="order_id", how="left")

In [ ]:
df_master = df_master.merge(products, on="product_id", how="inner")

In [ ]:
df_master = df_master.merge(customers, on="customer_id", how="left")

In [ ]:
df_master = df_master.merge(payments, on="order_id", how="left")

In [ ]:
df_master = df_master.merge(products, on="product_id", how="left")

In [ ]:
df_master = df_master.merge(sellers, on="seller_id", how="left")

In [ ]:
df_master = df_master.merge(review, on="order_id", how="left")

In [ ]:
dup_col = [c for c in df_master.columns if '_x' in c]

In [ ]:
dup_col


### Q no.3 Merge the product_category_name_translation table to replace Portuguese category names with English equivalents in df_master. How many unique English product categories exist? Create a new column category_en and drop the original Portuguese column.

In [ ]:
translation = dfs['product_category_name_translation.csv']

In [ ]:
dfs['product_category_name_translation.csv']

In [ ]:
df_master = df_master.merge(
    translation,
    on='product_category_name',
    how='left'
)

In [ ]:
df_master['category_en'] = df_master["product_category_name_english"]

In [ ]:
df_master = df_master.drop(columns=['product_category_name'])

In [ ]:
unique_categories = df_master['category_en'].nunique()
print("Number of unique English product categories:", unique_categories)

### Q no.4 Parse all timestamp columns (order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date) to datetime. Create a column delivery_days = actual days taken to deliver (from purchase to actual delivery). Create another column delivery_delay_days = actual delivery − estimated delivery (negative = early, positive = late).

In [ ]:
df_master["review_creation_date"].dtype

In [ ]:
df_master["order_purchase_timestamp"] = pd.to_datetime(df_master["order_purchase_timestamp"])
df_master["order_delivered_customer_date"] = pd.to_datetime(df_master["order_delivered_customer_date"])
df_master["order_estimated_delivery_date"] = pd.to_datetime(df_master["order_estimated_delivery_date"])

In [ ]:
df_master["delivery_days"] = (df_master["order_delivered_customer_date"] - df_master["order_purchase_timestamp"]).dt.days

In [ ]:
df_master["delivery_delay_days"] = (df_master["order_delivered_customer_date"] - df_master["order_estimated_delivery_date"]).dt.days

### Q no.5 Generate a comprehensive missing-value audit table for df_master showing: column name, missing count, missing percentage, and data type. Display only columns with >0% missing values sorted by percentage descending. Should a missing category_en be imputed or dropped? Justify your choice.

In [ ]:
audit = (
    df_master.isnull()
    .sum()
    .reset_index()
    .rename(columns={'index':'column', 0:'missing_count'})
)

In [ ]:
audit["missing_pct"] = (audit["missing_count"] / len(df_master)) * 100

In [ ]:
audit["dtype"] = df_master.dtypes.values

In [ ]:
audit = audit[audit["missing_count"]>0].sort_values("missing_pct", ascending=False)

In [ ]:
audit

### Q no.6 Identify and remove orders whose order_status is not 'delivered' from the analytical dataset. What percentage of orders are excluded? Create a clean DataFrame called df_clean containing only delivered orders with no null delivery dates. Report its final shape.

In [ ]:
total_ordes = len(df_master)

In [ ]:
df_clean = df_master[
    (df_master["order_status"] == "delivered")&
    (df_master["order_delivered_customer_date"].notnull())
]

In [ ]:
excluded_orders = total_ordes - len(df_clean)

In [ ]:
excluded_pct = (excluded_orders / total_ordes) * 100

In [ ]:
print(f"{round(excluded_pct, 2)} %")
df_clean.shape

### Q no.7 [HARD]Detect and handle duplicate order_id entries that arise from the multi-table merge (one order → many items). For revenue analysis, aggregate order-level metrics: total payment_value per order, total price per order, and count of items per order. Store in a DataFrame called df_orders_agg (one row per order). Explain why deduplication before aggregation gives wrong results.

In [ ]:
df_orders_agg = (
    df_clean.groupby("order_id")
    .agg(
        total_payments_valuem=("payment_value", "sum"),
        total_price=("price", "sum"),
        item_count=("order_item_id", "count"),
    )
    .reset_index()
)

In [ ]:
df_orders_agg

# Section B
#### Feature Engineering & Advanced Business Metrics

### Engineer a Revenue Per Category table: group df_clean by category_en and compute: total revenue (sum of price), order count, average order value (AOV), and average review score. Sort by total revenue descending. Display the top 15 categories and identify which category has the highest AOV.

In [ ]:
df_category = (
    df_clean.groupby("category_en")
    .agg(
        total_revenue = ("price", "sum"),
        order_count = ("order_id", "nunique"),
        avg_order_value = ("price", "mean"),
        avg_review = ("review_score", "mean")
    )
    .reset_index()
)

In [ ]:
df_category = df_category.sort_values("total_revenue", ascending=False)

In [ ]:
df_category.head(15)

In [ ]:
highest_aov_category = df_category.loc[df_category["avg_order_value"].idxmax(), "category_en"]

In [ ]:
print("Category with highest AOV:", highest_aov_category)
# full form of Aov =  Analysis of Variance 

### Q no.9 Create a Customer Segmentation feature using RFM logic on df_clean: compute for each customer — Recency (days since last purchase from a reference date of 2018-09-01), Frequency (number of orders), and Monetary (total spend). Use pd.qcut() to assign RFM scores (1–4) and create a combined RFM_segment label.

In [ ]:
# RFM (Recency, Frequency, Monetary)

rft_date = pd.to_datetime("2018-09-01")

rfm =( df_clean.groupby("customer_id")
    .agg(
        last_purchase = ("order_purchase_timestamp", "max"),
        frequency = ("order_id", "nunique"),
        monetary = ("payment_value", "sum")

    )
    .reset_index()
    )

In [ ]:
rfm["recency"] = (rft_date - rfm["last_purchase"]).dt.days

In [ ]:
rfm['R_score'] = pd.qcut(rfm['recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['monetary'], 4, labels=[1,2,3,4])


In [ ]:
rfm["RFM_segment'"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

In [ ]:
rfm["RFM_segment'"]

### Q no.10 Calculate Seller Performance Metrics: for each seller, compute total revenue, number of unique customers served, average delivery delay, average review score, and on-time delivery rate (% orders with delivery_delay_days ≤ 0). Identify the top 5 sellers by revenue and the bottom 5 by customer satisfaction score.

In [ ]:
df_seller_perf = (df_clean.groupby("seller_id")
                  .agg(
                      total_revenue = ("price", "sum"),
                      unique_customers =("customer_id", "nunique"),
                      avg_delivery_delay = ("delivery_delay_days", "mean"),
                      avg_review_score =('review_score', 'mean'),
                      on_time_rate = ("delivery_delay_days", lambda x: (x <= 0).mean() * 100)

                  ).reset_index()
)

In [ ]:
top5_revenue = df_seller_perf.sort_values('total_revenue', ascending=False).head(5)

bottom5_satisfaction = df_seller_perf.sort_values('avg_review_score', ascending=True).head(5)

In [ ]:
print("Top 5 Sellers by Revenue:")
top5_revenue

In [ ]:
print("\nBottom 5 Sellers by Customer Satisfaction:")
bottom5_satisfaction

### Q no.11 Analyse payment behaviour: what is the distribution of payment types (credit_card, boleto, voucher, debit_card)? For credit card payments, compute the distribution of payment_installments. What percentage of customers pay in more than 6 installments? Create a cross-tabulation of payment_type vs. top 5 product categories.

In [ ]:
payment_dist = df_clean['payment_type'].value_counts(normalize=True) * 100

print("Payment type distribution (%):")
round(payment_dist, 2)

In [ ]:
payment_installments = (
    df_clean.loc[df_clean['payment_type'] == 'credit_card', 'payment_installments']
    .value_counts(normalize=True) * 100
)
print("\nCredit card installments distribution (%):")
round(payment_installments, 2)

In [ ]:
# Percentage paying
pct_over6 = (
    (df_clean.loc[df_clean['payment_type'] == 'credit_card', 'payment_installments'] > 6).mean() * 100
)
print(f"{round(pct_over6, 2)} % percentage of customers pay more than 6 installments :)")

In [ ]:
top5_categories = (
    df_clean['category_en'].value_counts().head(5).index
)

crosstab = pd.crosstab(
    df_clean.loc[df_clean['category_en'].isin(top5_categories), 'payment_type'],
    df_clean.loc[df_clean['category_en'].isin(top5_categories), 'category_en']
)

print("\nPayment type vs Top 5 Categories:")
crosstab

### Q no.12 Compute Monthly Cohort Retention: define each customer's cohort as the month of their first purchase. For each cohort, compute what fraction of customers made a repeat purchase in months 1, 2, 3 after their first purchase. Display the result as a cohort retention matrix (cohort months as rows, period months as columns).

#### information
##### Cohort Retention in Coding with Pandas
###### Cohort retention analysis in coding with pandas involves grouping users based on shared characteristics within a defined time window and tracking their behavior over subsequent periods. This analysis helps businesses understand user retention trends and make informed decisions to improve user engagement and satisfaction.

In [ ]:
df_clean['order_month'] = df_clean['order_purchase_timestamp'].dt.to_period('M')

cohort = (
    df_clean.groupby('customer_id')['order_month']
    .min()
    .reset_index()
    .rename(columns={'order_month':'cohort_month'})
    )

# a group of people with a shared characteristic:

In [ ]:
df_cohort = df_clean.merge(cohort, on='customer_id')

In [ ]:
df_cohort["period_month"] = (
    (df_cohort["order_month"].dt.to_timestamp()
    - df_cohort["cohort_month"].dt.to_timestamp()).dt.days // 30
)

In [ ]:
cohort_counts = (
    df_cohort.groupby(['cohort_month','period_month'])['customer_id']
    .nunique()
    .reset_index()
)

In [ ]:
cohort_matrix = cohort_counts.pivot(
    index='cohort_month',
    columns='period_month',
    values='customer_id'
)

In [ ]:
cohort_size = cohort_matrix[0]
retention = cohort_matrix.divide(cohort_size, axis=0)

In [ ]:
available_periods = [p for p in [1,2,3] if p in retention.columns]
retention = retention[available_periods]

print("Monthly Cohort Retention (fraction of customers):")
retention

In [ ]:
available_periods = [p for p in [1,2,3] if p in retention.columns]

available_periods

### Q no.13 Engineer a product weight tier column: use NumPy's np.digitize() to classify products into 4 weight bands based on product_weight_g. Analyse whether heavier products have longer delivery times. Compute the Pearson correlation between weight and delivery_days using np.corrcoef().

In [ ]:
bins = [0, 500, 1000, 2000, np.inf]
labels = ['Light', 'Medium', 'Heavy', 'Very Heavy']

In [ ]:
bin_indices = np.digitize(df_clean['product_weight_g'], bins) - 1

In [ ]:
bin_indices = np.clip(bin_indices, 0, len(labels)-1)

In [ ]:
df_clean['weight_tier'] = [labels[i] for i in bin_indices]

In [ ]:
tier_analysis = (
    df_clean.groupby('weight_tier')['delivery_days']
    .mean()
    .reset_index()
    .rename(columns={'delivery_days':'avg_delivery_days'})
)

In [ ]:
print("Average delivery days by weight tier:")
round(tier_analysis, 2)

In [ ]:
corr_matrix = np.corrcoef(df_clean['product_weight_g'], df_clean['delivery_days'])
corr = corr_matrix[0,1]

print("Pearson correlation between product weight and delivery_days:", corr)

### Q no.14 [HARD]Implement Review Score Prediction Features: engineer at least 5 features that could predict review score (e.g., delivery delay, product weight, price, category, installments). Compute the correlation matrix for all numeric features against review_score using df.corr() and identify the top 3 strongest predictors. Visualise with a seaborn heatmap saved as a PNG.

In [ ]:
df_clean['delivery_delay_days'] = (
    df_clean['order_delivered_customer_date'] - df_clean['order_estimated_delivery_date']
).dt.days

In [ ]:
df_clean['delivery_days'] = (
    df_clean['order_delivered_customer_date'] - df_clean['order_purchase_timestamp']
).dt.days


In [ ]:
df_clean['product_weight_g'] = df_clean['product_weight_g'].fillna(0)

In [ ]:
df_clean['price'] = df_clean['price'].fillna(0)


In [ ]:
df_clean['payment_installments'] = df_clean['payment_installments'].fillna(0)

In [ ]:
category_map = df_clean['category_en'].value_counts().to_dict()
df_clean['category_freq'] = df_clean['category_en'].map(category_map)

In [ ]:
features = ['delivery_delay_days','delivery_days','product_weight_g',
            'price','payment_installments','category_freq','review_score']

corr_matrix = df_clean[features].corr()

print("Correlation matrix with review_score:")
corr_matrix['review_score'].sort_values(ascending=False)


In [ ]:
top3 = corr_matrix['review_score'].abs().sort_values(ascending=False).iloc[1:4]
print("Top 3 strongest predictors of review_score:")
top3


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap: Review Score Predictors")
plt.tight_layout()
plt.savefig("review_score_predictors.png")
plt.show()


# Section C
### Statistical Deep-Dive with NumPy & Time-Series Analysis

### q no. 15 Using only NumPy (no Pandas statistical methods), compute for the price column: mean, median, standard deviation, skewness (using the formula: 3*(mean-median)/std — Pearson's approximation), and kurtosis. Interpret whether the price distribution is approximately normal or highly skewed.

In [ ]:
prices = df_clean['price'].to_numpy()

mean_price = np.mean(prices)

median_price = np.median(prices)

std_price = np.std(prices, ddof=0)

skewness = 3 * (mean_price - median_price) / std_price

# Formula: (E[(X-μ)^4] / σ^4) - 3
kurtosis = np.mean(((prices - mean_price) ** 4)) / (std_price ** 4) - 3

print("Mean:", mean_price)
print("Median:", median_price)
print("Std Dev:", std_price)
print("Skewness:", skewness)
print("Kurtosis:", kurtosis)


### Q no. 16 Resample the order data to monthly frequency using the order_purchase_timestamp column. Compute: total monthly revenue, total monthly orders, and average monthly order value. Plot a dual-axis line chart showing revenue (left axis) and order count (right axis) from 2017-01 to 2018-08. Identify the peak sales month.

In [ ]:
df_clean['order_purchase_timestamp'] = pd.to_datetime(df_clean['order_purchase_timestamp'])

In [ ]:
monthly = (
    df_clean.set_index('order_purchase_timestamp')
    .resample('M')
    .agg(
        total_revenue=('price','sum'),
        total_orders=('order_id','nunique')
    )
)

In [ ]:
monthly['avg_order_value'] = monthly['total_revenue'] / monthly['total_orders']

In [ ]:
monthly_period = monthly.loc['2017-01':'2018-08']

fig, ax1 = plt.subplots(figsize=(10,6))

ax1.plot(monthly_period.index, monthly_period['total_revenue'], color='tab:blue', marker='o')
ax1.set_ylabel("Total Revenue", color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(monthly_period.index, monthly_period['total_orders'], color='tab:red', marker='s')
ax2.set_ylabel("Total Orders", color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title("Monthly Revenue vs Orders (2017-01 to 2018-08)")
plt.tight_layout()
plt.show()


In [ ]:
peak_month = monthly_period['total_revenue'].idxmax()
peak_value = monthly_period['total_revenue'].max()

print("Peak sales month:", peak_month.strftime("%Y-%m"), "with revenue:", peak_value)


### Q no.17 Perform an outlier analysis on price and freight_value using three methods: (a) c (|z| > 3), (b) IQR method (Q1−1.5×IQR to Q3+1.5×IQR), (c) Percentile clipping at 1st and 99th percentile. Compare how many outliers each method identifies. Which method is most appropriate for this dataset and why?

In [ ]:
#  IQR stands for Interquartile Range, a measure of the spread of the middle 50% of a dataset.

price = df_clean['price'].to_numpy()
freight = df_clean['freight_value'].to_numpy()

z_price = (price - np.mean(price)) / np.std(price)
z_freight = (freight - np.mean(freight)) / np.std(freight)

outliers_z_price = np.sum(np.abs(z_price) > 3)
outliers_z_freight = np.sum(np.abs(z_freight) > 3)


In [ ]:
def iqr_outliers(arr):
    Q1 = np.percentile(arr, 25)
    Q3 = np.percentile(arr, 75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    return np.sum((arr < lower) | (arr > upper))

outliers_iqr_price = iqr_outliers(price)
outliers_iqr_freight = iqr_outliers(freight)


In [ ]:
def percentile_outliers(arr):
    lower = np.percentile(arr, 1)
    upper = np.percentile(arr, 99)
    return np.sum((arr < lower) | (arr > upper))

outliers_pct_price = percentile_outliers(price)
outliers_pct_freight = percentile_outliers(freight)


In [ ]:
print("Price outliers:")
print("Z-score:", outliers_z_price)
print("IQR:", outliers_iqr_price)
print("Percentile:", outliers_pct_price)

print("\nFreight outliers:")
print("Z-score:", outliers_z_freight)
print("IQR:", outliers_iqr_freight)
print("Percentile:", outliers_pct_freight)


### Q no, 18 Analyse weekly order patterns: extract the day of the week from order_purchase_timestamp. Using NumPy, compute the mean and std of daily orders for each weekday. Create a polar bar chart or regular bar chart showing the busiest days. Are weekends significantly different from weekdays? Support with percentage difference.

In [ ]:
df_clean['order_purchase_timestamp'] = pd.to_datetime(df_clean['order_purchase_timestamp'])

df_clean['weekday'] = df_clean['order_purchase_timestamp'].dt.dayofweek

daily_orders = (
    df_clean.groupby(df_clean['order_purchase_timestamp'].dt.date)['order_id']
    .nunique()
    .reset_index(name='orders')
)

daily_orders['weekday'] = pd.to_datetime(daily_orders['order_purchase_timestamp']).dt.dayofweek


In [ ]:
weekday_stats = {}
for wd in range(7):
    arr = daily_orders.loc[daily_orders['weekday']==wd, 'orders'].to_numpy()
    weekday_stats[wd] = {
        'mean': np.mean(arr),
        'std': np.std(arr, ddof=0)
    }

print("Mean & Std of daily orders per weekday:")
print(weekday_stats)


In [ ]:
labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
means = [weekday_stats[wd]['mean'] for wd in range(7)]

plt.figure(figsize=(8,5))
plt.bar(labels, means, color='skyblue')
plt.title("Average Daily Orders by Weekday")
plt.ylabel("Mean Orders")
plt.tight_layout()
plt.show()


### Q no.19 Compute a 30-day rolling average of daily revenue using NumPy's np.convolve() with a uniform kernel. Overlay the rolling average on the raw daily revenue time-series. How does the rolling average help reveal the underlying trend vs. noise?

In [ ]:
df_clean['order_purchase_timestamp'] = pd.to_datetime(df_clean['order_purchase_timestamp'])

daily_revenue = (
    df_clean.groupby(df_clean['order_purchase_timestamp'].dt.date)['price']
    .sum()
    .reset_index(name='revenue')
)

rev = daily_revenue['revenue'].to_numpy()


In [ ]:
kernel = np.ones(30) / 30
rolling_avg = np.convolve(rev, kernel, mode='valid')

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(daily_revenue['order_purchase_timestamp'], rev, 
         color='lightgray', label='Daily Revenue')

plt.plot(daily_revenue['order_purchase_timestamp'][29:], rolling_avg, 
         color='blue', linewidth=2, label='30-day Rolling Avg')

plt.title("Daily Revenue with 30-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()


### Q no. 20 Build a State-Level Revenue Matrix using NumPy: create a 2D array where rows = top 10 states (by total revenue) and columns = top 5 product categories. Values = total revenue for each state-category combination. Normalise each row by its maximum value so values range from 0 to 1. Print the matrix with state labels.

In [ ]:
state_cat_rev = (
    df_clean.groupby(['customer_state','category_en'])['price']
    .sum()
    .reset_index()
)

top_states = (
    state_cat_rev.groupby('customer_state')['price']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top_categories = (
    state_cat_rev.groupby('category_en')['price']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

filtered = state_cat_rev[
    state_cat_rev['customer_state'].isin(top_states) &
    state_cat_rev['category_en'].isin(top_categories)
]


In [ ]:
matrix = np.zeros((len(top_states), len(top_categories)))

for i, state in enumerate(top_states):
    for j, cat in enumerate(top_categories):
        val = filtered.loc[
            (filtered['customer_state']==state) & (filtered['category_en']==cat),
            'price'
        ].sum()
        matrix[i,j] = val


In [ ]:
row_max = matrix.max(axis=1).reshape(-1,1)
norm_matrix = matrix / row_max


In [ ]:
print("State-Level Revenue Matrix (Normalised 0–1):\n")
for i, state in enumerate(top_states):
    row_vals = "  ".join([f"{norm_matrix[i,j]:.2f}" for j in range(len(top_categories))])
    print(f"{state}: {row_vals}")


### Q no.21 [HARD]Implement a Monte Carlo simulation to estimate the 95% confidence interval of the true mean order value. Generate 10,000 bootstrap samples (with replacement) of size 1,000 from the price column using np.random.choice(). Compute the mean of each sample. Plot the bootstrap distribution and shade the 2.5th–97.5th percentile region.

In [ ]:
prices = df_clean['price'].to_numpy()

n_bootstrap = 10000
sample_size = 1000

bootstrap_means = []
for _ in range(n_bootstrap):
    sample = np.random.choice(prices, size=sample_size, replace=True)
    bootstrap_means.append(np.mean(sample))

bootstrap_means = np.array(bootstrap_means)


In [ ]:
# 95% CI = 2.5th and 97.5th percentiles
lower = np.percentile(bootstrap_means, 2.5)
upper = np.percentile(bootstrap_means, 97.5)

print("95% Confidence Interval for Mean Order Value:")
print(f"Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")


In [ ]:
plt.figure(figsize=(10,6))
plt.hist(bootstrap_means, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(lower, color='red', linestyle='--', label=f"2.5th percentile ({lower:.2f})")
plt.axvline(upper, color='red', linestyle='--', label=f"97.5th percentile ({upper:.2f})")
plt.title("Bootstrap Distribution of Mean Order Value")
plt.xlabel("Mean Order Value")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
cohort_matrix.to_csv("../data/cohort_matrix.csv", index=False)

In [ ]:
cohort_matrix.head()